In [ ]:
"""
Core Capability -- Contact NCBI and pull data
"""
from time import sleep
import re
from collections import defaultdict
from io import BytesIO, StringIO
from pathlib import Path
from zipfile import ZipFile
import matplotlib.pyplot as plt
from Bio import AlignIO, Phylo, SeqIO
from Bio.Align.Applications import MuscleCommandline
from Bio.Phylo.TreeConstruction import (
    NNITreeSearcher,
    ParsimonyScorer,
    ParsimonyTreeConstructor,
)
from Bio.SeqRecord import SeqRecord
from ncbi.datasets import GeneApi
from GeneClasses import NaturalGene, ProteinObj, Isoform

def determineProtWeight(aaSeq: str):
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    prot = ProteinAnalysis(aaSeq)
    return prot.molecular_weight()

def checkAgainstProteins(transDict: dict, proteins: list[ProteinObj]):
    #Find the seq:aa pair in the dict that matches with the proteins from the NCBI data
    # transdict is a dictionary of mRNA sequences (keys) and their translations (values)
    # proteins is a list of ProteinObj objects
        # ProteinObj is a dataclass with the following attributes:
            # aaSeq: str
            # protWeight: float
            # associatedGeneID: int
    theTrans = ''
    therna = ''
    theProt = None
    for translation in transDict.values():
        for protein in proteins:
            if translation == protein.aaSeq:
                theTrans = translation
                theProt = protein
                therna = [k for k, v in transDict.items() if v == theTrans][0]
                break
    if theProt == None:
        return None
    else:
        return [therna, theTrans, theProt]


def saveRefGene(record):
    import pickle
    import os

    name = str(record.id) + r'.fasta'
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\ReferenceGenes'
    fname = os.path.join(root, name)
    print(fname)
    with open(fname, 'wb') as outfile:
        pickle.dump(record, outfile)
    outfile.close()

def translatemRNA(mRNASeq: str):
    from Bio.Seq import Seq
    mRNASeq = mRNASeq.upper()
    newStr = ''
    for letter in mRNASeq:
        if letter == 'U':
            newStr += 'T'
        else:
            newStr += letter
    strToFind = r'ATG'
    startVec = []
    seqsVec = []
    transVec = []
    for i in range(len(newStr) - 2):
        if newStr[i:i+3] == strToFind:
            startVec.append(i)
    for starter in startVec:
        seqsVec.append(newStr[starter:])
    for subStr in seqsVec:
        nSeq = Seq(subStr)
        transVec.append(nSeq.translate(to_stop=True))
    return dict(zip(seqsVec, transVec))

def gatherDataonReferenceGeneHuman(geneID):
    #Take a gene ID and run it through the NCBI API to gather data on the gene
    #This data will be used to create a reference gene database
    #This database will be used to compare to the written gene to determine the best CDS
    from Bio import Entrez
    from Bio import SeqIO
    import re

    #geneID = int(geneID)
    Entrez.email = "lukeontheearth@gmail.com"
    handle = Entrez.efetch(db="protein", id=geneID, rettype="fasta_cds_na", retmode="xml")
    record = handle.read()
    record = re.sub('\\n\\n', '\\n', record)
    #record = SeqIO.read(handle, "genbank")
    print(record)
    saveRefGene(record)
    handle.close()

def dataByNameHumanSpecPath(geneName, saveToPath):
    # Take a gene ID and run it through the NCBI API to gather data on the gene
    # This data will be used to create a reference gene database
    # This database will be used to compare to the written gene to determine the best CDS
    from Bio import Entrez
    from Bio import SeqIO
    import pickle
    import os
    from Bio.SeqRecord import SeqRecord
    import re
    #must be one of ['FASTA_UNSPECIFIED', 'FASTA_GENE', 'FASTA_RNA', 'FASTA_PROTEIN', 'FASTA_GENE_FLANK', 'FASTA_CDS', 'FASTA_5P_UTR', 'FASTA_3P_UTR']

    geneIDs = get_gene_ids(geneName, "human")
    print(str(geneIDs))
    justIDs = list(geneIDs.values())

    #Check if we already have data for the gene and remove from the list if so
    try:
        path = download_transcripts(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHolding')
        sleep(1)
        pPath = download_proteins(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHoldingP')
        sleep(1)
        gPath = download_genomicDNA(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHoldingG')
    except KeyError:
        print("Error downloding data for gene " + geneName[0] + ".")
        return None

    for ID in justIDs:
        recs = SeqIO.parse(path, 'fasta')
        gRecs = SeqIO.parse(gPath, 'fasta')
        pRecs = SeqIO.parse(pPath, 'fasta')
        sString = r'[GeneID=' + str(ID) + r']'
        isoVec = []
        protVec = []
        gSeq = ''
        gName = ''
        chromosome = -1
        gList = geneIDs.keys()
        for g in gList:
            if geneIDs[g] == ID:
                gName = g
                break

        # >NC_000021.9:44285876-44298648 AIRE [organism=Homo sapiens] [GeneID=326] [chromosome=21]

        for gRec in gRecs:
            if sString in gRec.description:
                gSeq = gRec.seq
                chromosomeStart = int(gRec.description.find(r'[chromosome=') + 12)
                i = 0
                while True:
                    char = gRec.description[chromosomeStart + i]
                    if char == r']':
                        break
                    i += 1
                chromosomeEnd = chromosomeStart + i
                try:
                    chromosome = int(gRec.description[chromosomeStart:chromosomeEnd])
                except ValueError:
                    chromosome = 0
                except KeyError:
                    continue
                #print(str(chromosome))
                #add chromosome number to the gene object
                #dd the locus start stop to the gene object

        for pRec in pRecs:
            if sString in pRec.description:
                aaSeq = str(pRec.seq)
                pWeight = determineProtWeight(aaSeq)
                pObj = ProteinObj(aaSeq, pWeight, ID)
                protVec.append(pObj)
        tempVec = []
        for p in protVec:
            if p not in tempVec:
                tempVec.append(p)
        protVec = tempVec

        for rec in recs:
            if sString in rec.description:
                if r'[transcript=' in rec.description:
                    ssStr = str(rec.description)
                    starting = ssStr.find(r'[transcript=') + 12
                    i = 0
                    for char in ssStr[starting:]:
                        if char == r']':
                            break
                        i += 1
                    stopping = starting + i
                    isoNum = ssStr[starting:stopping]
                    print("Gene name: " + gName + " isoform number: " + isoNum)
                    rnaTransDict = translatemRNA(str(rec.seq))
                    corr = checkAgainstProteins(rnaTransDict, protVec)
                    if corr == None:
                        print('No match found for the protein object')
                    else:
                        corrProtObj = corr[2]
                        corrAASeq = corr[1]
                        corrRNA = corr[0]
                        if corrProtObj.aaSeq == corrAASeq:
                            newIso = Isoform(isoNum, corrProtObj, rec.seq, corrRNA, -1)
                            isoVec.append(newIso)
                        else:
                            print('AA seq of the protein object does not match the aa seq of the NCBI protein')
                            continue
                else:
                    print("Gene name: " + gName + " sole isoform.")
                    rnaTransDict = translatemRNA(str(rec.seq))
                    corr = checkAgainstProteins(rnaTransDict, protVec)
                    if corr == None:
                        print('No match found for the protein object')
                    else:
                        corrProtObj = corr[2]
                        corrAASeq = corr[1]
                        corrRNA = corr[0]
                        if corrProtObj.aaSeq == corrAASeq:
                            newIso = Isoform(1, corrProtObj, rec.seq, corrRNA, -1)
                            isoVec.append(newIso)
                        else:
                            print('AA seq of the protein object does not match the aa seq of the NCBI protein')
                            continue
        newGene = NaturalGene(isoVec, ID, gName, 'homo sapien', gSeq, chromosome)

        name = str(newGene.geneID) + newGene.geneName + r'.pkl'
        root = saveToPath
        fname = os.path.join(root, name)
        with open(fname, 'wb') as infile:
            pickle.dump(newGene, infile)
        infile.close()
        saveNaturalGeneObj(newGene)




def saveNaturalGeneObj(obj: NaturalGene):
    import pickle
    import os
    name = str(obj.geneID) + obj.geneName + r'.pkl'
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\ReferenceGenes'
    fname = os.path.join(root, name)
    with open(fname, 'wb') as infile:
        pickle.dump(obj, infile)
    infile.close()


def dataByNameHuman(geneName):
    # Take a gene ID and run it through the NCBI API to gather data on the gene
    # This data will be used to create a reference gene database
    # This database will be used to compare to the written gene to determine the best CDS
    from Bio import Entrez
    from Bio import SeqIO
    from Bio.SeqRecord import SeqRecord
    import re
    #must be one of ['FASTA_UNSPECIFIED', 'FASTA_GENE', 'FASTA_RNA', 'FASTA_PROTEIN', 'FASTA_GENE_FLANK', 'FASTA_CDS', 'FASTA_5P_UTR', 'FASTA_3P_UTR']

    geneIDs = get_gene_ids(geneName, "human")
    print(str(geneIDs))
    justIDs = list(geneIDs.values())

    #Check if we already have data for the gene and remove from the list if so

    path = download_transcripts(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHolding')
    sleep(1)
    pPath = download_proteins(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHoldingP')
    sleep(1)
    gPath = download_genomicDNA(justIDs, r'C:\Users\Luke\PycharmProjects\GeneRider\NCBIHoldingG')

    for ID in justIDs:
        recs = SeqIO.parse(path, 'fasta')
        gRecs = SeqIO.parse(gPath, 'fasta')
        pRecs = SeqIO.parse(pPath, 'fasta')
        sString = r'[GeneID=' + str(ID) + r']'
        isoVec = []
        protVec = []
        gSeq = ''
        gName = ''
        chromosome = -1
        gList = geneIDs.keys()
        for g in gList:
            if geneIDs[g] == ID:
                gName = g
                break

        # >NC_000021.9:44285876-44298648 AIRE [organism=Homo sapiens] [GeneID=326] [chromosome=21]

        for gRec in gRecs:
            if sString in gRec.description:
                gSeq = gRec.seq
                chromosomeStart = int(gRec.description.find(r'[chromosome=') + 12)
                i = 0
                while True:
                    char = gRec.description[chromosomeStart + i]
                    if char == r']':
                        break
                    i += 1
                chromosomeEnd = chromosomeStart + i
                chromosome = int(gRec.description[chromosomeStart:chromosomeEnd])
                #print(str(chromosome))
                #add chromosome number to the gene object
                #dd the locus start stop to the gene object

        for pRec in pRecs:
            if sString in pRec.description:
                aaSeq = str(pRec.seq)
                pWeight = determineProtWeight(aaSeq)
                pObj = ProteinObj(aaSeq, pWeight, ID)
                protVec.append(pObj)
        tempVec = []
        for p in protVec:
            if p not in tempVec:
                tempVec.append(p)
        protVec = tempVec

        for rec in recs:
            if sString in rec.description:
                if r'[transcript=' in rec.description:
                    ssStr = str(rec.description)
                    starting = ssStr.find(r'[transcript=') + 12
                    i = 0
                    for char in ssStr[starting:]:
                        if char == r']':
                            break
                        i += 1
                    stopping = starting + i
                    isoNum = ssStr[starting:stopping]
                    print("Gene name: " + gName + " isoform number: " + isoNum)
                    rnaTransDict = translatemRNA(str(rec.seq))
                    corr = checkAgainstProteins(rnaTransDict, protVec)
                    if corr == None:
                        print('No match found for the protein object')
                    else:
                        corrProtObj = corr[2]
                        corrAASeq = corr[1]
                        corrRNA = corr[0]
                        if corrProtObj.aaSeq == corrAASeq:
                            newIso = Isoform(isoNum, corrProtObj, rec.seq, corrRNA, -1)
                            isoVec.append(newIso)
                        else:
                            print('AA seq of the protein object does not match the aa seq of the NCBI protein')
                            continue
                else:
                    print("Gene name: " + gName + " sole isoform.")
                    rnaTransDict = translatemRNA(str(rec.seq))
                    corr = checkAgainstProteins(rnaTransDict, protVec)
                    if corr == None:
                        print('No match found for the protein object')
                    else:
                        corrProtObj = corr[2]
                        corrAASeq = corr[1]
                        corrRNA = corr[0]
                        if corrProtObj.aaSeq == corrAASeq:
                            newIso = Isoform(1, corrProtObj, rec.seq, corrRNA, -1)
                            isoVec.append(newIso)
                        else:
                            print('AA seq of the protein object does not match the aa seq of the NCBI protein')
                            continue
        newGene = NaturalGene(isoVec, ID, gName, 'homo sapien', gSeq, chromosome)
        saveNaturalGeneObj(newGene)


def pullGeneByID(ID):
    gene_api = GeneApi()
    isoform1transcript = gene_api.download_gene_package_endpoint
    print(isoform1transcript)
    print(str(gene_api))

# https://github.com/ncbi/workshop-ncbi-data-with-python/blob/main/notebooks/workshop.py
def get_gene_ids(symbols, taxon):
    gene_api = GeneApi()
    gene_metadata = gene_api.gene_metadata_by_tax_and_symbol(symbols, taxon)
    gene_id_dict = {
        gene_data.gene.symbol: int(gene_data.gene.gene_id)
        for gene_data in gene_metadata.genes
    }

    return gene_id_dict

def query_orthologs(gene_id, taxa):
    gene_api = GeneApi()
    response = gene_api.gene_orthologs_by_id(gene_id, taxon_filter=taxa)
    return [item.gene for item in response.genes.genes]

def download_transcripts(gene_ids, data_directory):
    # Create a GeneApi object
    gene_api = GeneApi()
    print("Begin download of data package ...")
    # Use the GeneApi to download a fasta file of gene transcripts.
    gene_ds_download = gene_api.download_gene_package(
        gene_ids, include_annotation_type=["FASTA_RNA"], _preload_content=False
    )
    # The downloaded data package is formatted as a zip file.
    # Extract the fasta file and write it to your hard drive.
    with ZipFile(BytesIO(gene_ds_download.data)) as zipfile:
        data_file_name = zipfile.extract(
            "ncbi_dataset/data/rna.fna", path=data_directory
        )
    print(f"Download completed -- see {data_file_name}")
    del(gene_api)
    # Return the path to the fasta file you just downloaded.
    return Path(data_file_name)

def download_proteins(gene_ids, data_directory):
    # Create a GeneApi object
    gene_api = GeneApi()
    print("Begin download of data package ...")
    # Use the GeneApi to download a fasta file of gene transcripts.
    gene_ds_download = gene_api.download_gene_package(
        gene_ids, include_annotation_type=["FASTA_PROTEIN"], _preload_content=False
    )
    # The downloaded data package is formatted as a zip file.
    # Extract the fasta file and write it to your hard drive.
    with ZipFile(BytesIO(gene_ds_download.data)) as zipfile:
        data_file_name = zipfile.extract(
            "ncbi_dataset/data/protein.faa", path=data_directory
        )
    print(f"Download completed -- see {data_file_name}")
    del(gene_api)
    # Return the path to the fasta file you just downloaded.
    return Path(data_file_name)

def download_genomicDNA(gene_ids, data_directory):
    # Create a GeneApi object
    # import os
    gene_api = GeneApi()
    print("Begin download of data package ...")
    # Use the GeneApi to download a fasta file of gene transcripts.
    # ['FASTA_UNSPECIFIED', 'FASTA_GENE', 'FASTA_RNA', 'FASTA_PROTEIN', 'FASTA_GENE_FLANK', 'FASTA_CDS', 'FASTA_5P_UTR', 'FASTA_3P_UTR']
    gene_ds_download = gene_api.download_gene_package(
        gene_ids, include_annotation_type=["FASTA_GENE"], _preload_content=False, filename="genomicDNA.fna"
    )
    """Args:
            gene_ids ([int]): NCBI gene ids

        Keyword Args:
            include_annotation_type ([V1Fasta]): Select additional types of annotation to include in the data package.  If unset, no annotation is provided.. [optional]
            fasta_filter ([str]): Limit the FASTA sequences in the datasets package to these transcript and protein accessions. [optional]
            filename (str): Output file name.. [optional] if omitted the server will use the default value of "ncbi_dataset.zip"
            _return_http_data_only (bool): response data without head status
                code and headers. Default is True.
            _preload_content (bool): if False, the urllib3.HTTPResponse object
                will be returned without reading/decoding response data.
                Default is True.
            _request_timeout (int/float/tuple): timeout setting for this request. If
                one number provided, it will be total request timeout. It can also
                be a pair (tuple) of (connection, read) timeouts.
                Default is None.
            _check_input_type (bool): specifies if type checking
                should be done one the data sent to the server.
                Default is True.
            _check_return_type (bool): specifies if type checking
                should be done one the data received from the server.
                Default is True.
            _host_index (int/None): specifies the index of the server
                that we want to use.
                Default is read from the configuration.
            async_req (bool): execute request asynchronously

        Returns:
            file_type
                If the method is called asynchronously, returns the request
                thread.
     """
    # The downloaded data package is formatted as a zip file.
    # Extract the fasta file and write it to your hard drive.
    with ZipFile(BytesIO(gene_ds_download.data)) as zipfile:
        """data_file_name = zipfile.extract(
            "ncbi_dataset/data/rna.fna", path=data_directory
        )"""
        data_file_name = zipfile.extract(
            "ncbi_dataset/data/gene.fna", path=data_directory
        )
    print(f"Download completed -- see {data_file_name}")
    del(gene_api)
    # Return the path to the fasta file you just downloaded.
    return Path(data_file_name)

def get_organism_name(record):
    # Use re.search to match the pattern and capture the result.
    match = re.search(r"\[organism=(?P<name>[\w ]+)\]", record.description)
    # If there's a match, return the organism name.
    if match:
        return match.group("name")
    # If not, indicate that no name was found.
    else:
        return "OrganismNameNotFound"


def records_by_organism(records):
    # defaultdict is like a dict but with a default value for missing keys
    # here the default is an empty list, i.e., no records
    organism_dict = defaultdict(list)
    for record in records:
        org = get_organism_name(record)
        # Add the record to the list for its organism
        organism_dict[org].append(record)
    return organism_dict

def record_length(rec):
    return len(rec.seq)

"""
def get_longest_transcripts(record_dict):
    return {
        org: max(records, key=record_length)
        for org, records in organism_records.items()
    }
"""

def cds_region(transcript):
    # The range object is a list to account for the possibility
    # of multiple ranges. `[0]` takes the first one.
    cds_range = transcript.cds.range[0]
    # `(a, b)` is a pair of values. We're returning two things: begin and end
    return (int(cds_range.begin), int(cds_range.end))


def get_cds_regions(gene_list):
    return {
        transcript.accession_version: cds_region(transcript)
        # Using multiple `for`s loops over nested lists:
        for gene in gene_list
        for transcript in gene.transcripts
    }

def get_cds_records(transcript_dict, cds_regions):
    cds_records = []
    for organism, record in transcript_dict.items():
        # 1.
        start, end = cds_regions[record.id]
        #                2.
        cds_record = SeqRecord(
            #                 3.
            id=organism.replace(" ", "_"),
            name=record.name,
            description=record.description,
            #                     4.
            seq=record.seq[start - 1 : end],
        )
        cds_records.append(cds_record)
    return cds_records


def check_start_codons(proteins):
    return all([p.startswith("M") for p in proteins])


def check_stop_codons(proteins):
    return all([p.endswith("*") for p in proteins])

def align_with_muscle(input_fasta):
    muscle_exe = Path("../bin/muscle3.8.31_i86linux64")
    muscle_cline = MuscleCommandline(muscle_exe, input=input_fasta)
    # The variable `stdout` ("standard out") captures the output from MUSCLE
    # `stderr` ("standard error") captures any errors.
    stdout, stderr = muscle_cline()
    # `AlignIO` reads an alignment
    # `StringIO` lets BioPython treat a string as though it were a file
    return AlignIO.read(StringIO(stdout), "fasta")

def build_parsimony_tree(alignment):
    scorer = ParsimonyScorer()
    searcher = NNITreeSearcher(scorer)
    constructor = ParsimonyTreeConstructor(searcher)
    return constructor.build_tree(alignment)




def main():
    print("Let's pull some NCBI data")
    geneID = input("What gene ID should we pull?")
    try:
        pullGeneByID(geneID)
    except:
        print("Error")

if __name__ == '__main__':
    main()


"""
gene_symbols = ["MB", "BRCA1", "TP53"]
#                 8.               9.
gene_ids = get_gene_ids(gene_symbols, "human")
#        10.
print(gene_ids)
"""

